# parameter-tuning -- systematic tuning of eat-rest-v1 (baseline untouched)

Runs `external/candidates/tune_v1.py`: local random search around v1's shipped settings with successive halving.
The baseline file is never edited; every variant is the baseline's own policy class built with keyword overrides.

| Stage | Who | Seeds | Games |
|---|---|---|---|
| 1 | baseline + 80 variants (each changes 2-4 of 27 numeric settings) | 5000-5007 | 648 |
| 2 | baseline + best 12 | 6000-6023 (new) | 312 |
| 3 | baseline + best 3 | 7000-7063 (new) | 256 |

Fair comparison: every variant plays exactly the same seeds as the baseline and is scored as the paired difference
per seed. Winners are re-measured on fresh seeds at each stage, so only the FINAL table is an honest estimate. The
script ends with a verdict: `ADOPT` only if the best variant's gain exceeds twice its standard error on the 64 fresh
seeds, otherwise keep v1. Output: `logs/tune_v1/best.json`.

About 3 hours on 40 CPUs. It is launched detached, so it survives closing the laptop or the kernel, and it is
resumable: running the launch cell again continues from the finished games. Nothing else should use the CPUs meanwhile.
Mechanism study: nothing is written to `results/`.

**Cluster setup:** same as `test-eat-rest-v1.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [1]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [2]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 31 (delta 18), reused 22 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (31/31), 33.31 KiB | 1.96 MiB/s, done.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
   2ad60ab..ec433c1  challenge-1V2 -> origin/challenge-1V2
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is behind 'origin/challenge-1V2' by 4 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Updating 2ad60ab..ec433c1
Fast-forward
 .../eat-rest-selective/survival_agent.py           | 122 ++++
 .../external/candidates/sweep_config.py            |  65 +-
 .../external/candidates/tune_v1.py                 | 143 +++++
 .../survival-simulator

In [3]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## Launch (runs in the background)

In [4]:
import subprocess, sys
subprocess.Popen(f"mkdir -p logs && nohup {sys.executable} -u external/candidates/tune_v1.py --out logs/tune_v1 > logs/tune_v1.log 2>&1 &", shell=True)
print("started in the background -> logs/tune_v1.log")

started in the background -> logs/tune_v1.log


## Progress and results (rerun any time)

In [57]:
# Progress / results so far (rerun any time)
!grep -v "pkg_resources\|pygame\|^{" logs/tune_v1.log | tail -n 40

c037       1457    +111   128       6   29.4  {'population_decay': 959.628, 'dispersal_time': 8.951}
c039       1454    +107   119       7   30.4  {'rest_radius': 50.463, 'dispersal_time': 9.846, 'retirement': 95.042, 'search_speed': 0.325}
c006       1451    +105   109       5   26.9  {'harvest_range': 190.953, 'rest_radius': 59.46, 'reserve': 65.086, 'population_decay': 946.272}
c003       1449    +102   157       5   30.6  {'young_age': 69.054, 'population': 6, 'dispersal_distance': 107.152}
c036       1444     +97   174       6   25.2  {'maximum_birth_gap': 22.794, 'harvest_switch': 8.831, 'appetite_release': 58.539}
c021       1443     +96   112       5   29.0  {'emergency_fraction': 0.363, 'retirement': 101.668, 'appetite_level': 292.727, 'young_age': 71.118}
c029       1436     +90   112       4   30.0  {'population_decay': 908.683, 'crowd_margin': 36.836, 'retirement': 102.022, 'population': 5}
c055       1430     +83   147       5   28.4  {'idle_energy': 0.77, 'retirement': 11

In [9]:
# When finished: the verdict and the winning settings
import json, os
print(json.dumps(json.load(open("logs/tune_v1/best.json")), indent=1) if os.path.exists("logs/tune_v1/best.json") else "not finished yet")

not finished yet


In [42]:
!pgrep -af tune_v1.py; echo "---"; wc -l logs/tune_v1/games.jsonl; ls -la --time-style=full-iso logs/tune_v1/; date; echo "---"; tail -n 5 logs/tune_v1.log

15846 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19930 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19931 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19932 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19933 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19934 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19935 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19936 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19937 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19938 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19939 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19940 /opt/conda/bin/python -u external/candidates/tune_v1.py --out logs/tune_v1
19941 /opt/conda/bin/python 